In [ ]:
import gpytorch
import math
import torch
from matplotlib import pyplot as plt
import numpy as np

# %matplotlib inline
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from IPython.display import display, Math, Latex

# Import my own functions
from utils import upscale_tensor, minmax_normalise_tensor

# Scene data

In [ ]:
# Load tensors
hr_scene_bed_tensor = torch.load("./torch_data/scene_bed_tensor.pt")
hr_scene_sur_tensor = torch.load("./torch_data/scene_sur_tensor.pt")

In [ ]:
# Create low-res tensor of bed

factor = 5
# use utils function
lr_scene_bed_tensor = upscale_tensor(hr_scene_bed_tensor, upscaling_factor = factor)

print("HR tensor shape:", hr_scene_bed_tensor.shape)
print("LR tensor shape:", lr_scene_bed_tensor.shape)

In [ ]:
fig = px.imshow(lr_scene_bed_tensor.squeeze(), 
                color_continuous_scale = 'RdBu_r',
                origin = "upper", 
                title = "LR bed scene based on upsampling")
fig.show()

In [ ]:
fig = make_subplots(rows = 1, cols = 2,
                    subplot_titles = ("low resolution (lr) bed topography", "high resolution (hr) bed topography"))

fig.add_trace(go.Heatmap(z = lr_scene_bed_tensor.squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_scene_bed_tensor.squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 2)

fig.update_layout(plot_bgcolor = 'rgba(0,0,0,0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

In [ ]:
fig = make_subplots(rows = 1, cols = 3,
                    subplot_titles = ("lr bed", "hr bed", "hr surface"))

fig.add_trace(go.Heatmap(z = lr_scene_bed_tensor.squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_scene_bed_tensor.squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 2)

fig.add_trace(go.Heatmap(z = hr_scene_sur_tensor.squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 3)

fig.update_layout(autosize = False, height = 400, width = 900)
fig.update_layout(plot_bgcolor = 'rgba(0,0,0,0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

Observations:

- While the non-stationarities (boundries seem to be rather aliged, some surface artifacts are not visible in the bedrock.)

We will use HR surface to increase the resolution of the bedrock data.

## K_p covariance term

- Pixel intensity covariance function.
- Coupling/decoupling of pixels based on HR context.
- RBF (Squared exponential) kernel without output scale. Distance is calculated only over pixel values:  
 We can flatten the image channel (or channels) as this has no spatial meaning

### Visualise for last row

In [ ]:
mask = torch.ones(size = hr_scene_bed_tensor.squeeze().shape) * torch.nan
mask[-1, :] = torch.ones(size = (hr_scene_bed_tensor.squeeze().shape[0],))

fig = go.Figure(go.Heatmap(z = hr_scene_bed_tensor.squeeze() * mask, colorscale = 'haline'))
fig.update_layout(autosize = False, height = 600, width = 600)
fig.update_yaxes(autorange = "reversed")
fig.show()

In [ ]:
# Standardise to get sensitivity of default hp's
scaled_hr_scene_bed_tensor = minmax_normalise_tensor(hr_scene_bed_tensor)

In [ ]:
"""
### GPytorch covariance function###
kp_covar_module = gpytorch.kernels.RBFKernel()
# This kernel does not have an outputscale parameter - just like in paper

lazy_covar_matrix = kp_covar_module(scaled_hr_scene_bed_tensor[44, :]) # Returns a RootLinearOperator
kp_covar_matrix = lazy_covar_matrix.to_dense() # Gets the actual tensor for this kernel matrix
"""

## Kp covariance

Pixel-intensity covariance function. Referred to as either Radial Basis Function (RBF) kernel, Squared Exponention Kernel, or Gaussian kernel.   
We can't extrapolate more than lambda_p away from the data. Stationary. 

In [ ]:
# Write my own 1D RBF
def kp_covariance_function(tensor, lambda_p = 0.6):
    """ calculate pixel-intensity RBF covariance function.

    Args:
        tensor (torch.tensor): 2D or 1D tensor but with only one variable
        lambda_p (float, optional): Defaults to 0.6.

    Returns:
        _type_: rbf covariance 
    """
    # Assuming 1D
    # If tensor is not flat, flatten
    if len(list(tensor.shape)) > 1:
        tensor = tensor.reshape(-1)
    # broadcast shape to calculate pariwise distances
    dist = tensor.unsqueeze(-1) - tensor.unsqueeze(-2)
    # direction of distance does not matter
    dist_sqr = torch.pow(dist, exponent = 2)
    dist_sqr_scaled = dist_sqr/(2 * torch.pow(torch.tensor(lambda_p), exponent = 2))
    # 1 at zero distance
    rbf = torch.exp(- dist_sqr_scaled)
    return rbf

In [ ]:
# Detach from computational graph
fig = go.Figure(go.Heatmap(z = kp_covariance_function(scaled_hr_scene_bed_tensor[44, :], lambda_p = 0.6), colorscale = 'haline'))
fig.update_layout(autosize = False, height = 600, width = 600, title = "Covaraince matrix for last row of high resolution bed")
fig.update_yaxes(autorange = "reversed")
fig.show()

## K_s covariance term

In [ ]:
# arange function exclude the last one: (1, 46) goes up to (not including) 46
xs = torch.arange(1, 46).repeat(45,1)
ys = xs.T

# Divide by 45 to standardise
dims = 45
### Create mid_point data set for spatial kernel ###
mid_points = torch.cat((ys.unsqueeze(0), xs.unsqueeze(0)), dim = 0)/dims

In [ ]:
def ks_covariance_function(tensor, lambda_value = 0.2):
    """ spatial covariance function

    Args:
        tensor (_type_): _description_
        lambda_s (float, optional): _description_. Defaults to 0.2.

    Returns:
        _type_: _description_
    """
    # If tensor is not flat, flatten
    if len(list(tensor.shape)) > 2:
        # 2 is hardcoded
        tensor = tensor.reshape(2, -1)

    ### Euclidean distance (2D) ###
    # broadcast and calculate pairwise distances 
    dist = tensor.unsqueeze(-1) - tensor.unsqueeze(-2)
    dist_sqr = torch.pow(dist, exponent = 2)
    # sum across x and y axis
    dist_sum = torch.sum(dist_sqr, dim = 0)
    # take sqrt
    euc_dist = torch.sqrt(dist_sum)
    
    Z = torch.div(euc_dist, lambda_value)

    # Mask large Z's with nan before replacing values with zero
    Z[Z >= 1] = float('nan')

    # first term pushes small distanced to 0 and distances near 1 close to zero
    # second terms is clipped at 1 so that small distances will approach 1
    cov_matrix = torch.pow((1 - Z), exponent = 3) * ((3 * Z) + 1)
    cov_matrix[torch.isnan(cov_matrix)] = 0.0

    return cov_matrix

In [ ]:
# Visualise mapping
Z = torch.tensor(np.linspace(0, 1, 100))
cov_matrix = torch.pow(1. - Z, exponent = 3) * ((3 * Z) + 1)

px.scatter(x = Z.numpy(), y = cov_matrix.numpy(), title = "Mapping of Z").update_layout(
    xaxis_title = "Z value", yaxis_title = "covariance")

### Visualise spatial covariance for one row

In [ ]:
# Subset last row
mid_points_last_row = mid_points.reshape(2, -1)[:, -45:]

# Apply function
ks_covar_matrix = ks_covariance_function(mid_points_last_row, lambda_value = 0.2)

In [ ]:
def ks_kp_covar(lambda_s = 0.3, lambda_p = 0.4, sigma_f = 1.0, rowcol_index_tuple = (44, None), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor):
    # Subset rows/columns
    (row_index, column_index) = rowcol_index_tuple
    
    if (column_index == None):
        ks_input = mid_points[:, row_index, :]
        kp_input = kp_ds[row_index, :]
    else:
        ks_input = mid_points[:, :, column_index]
        kp_input = kp_ds[:, column_index]

    ### Ks ### smooth, sparse, local
    # lambda_s is the hp that controls the receptive field
    # ks input
    mid_points
    ks_covar_matrix = ks_covariance_function(ks_input, lambda_value = lambda_s)


    ### Kp ### coupling(decoupling) of similar(dissimilar) pixel values, non-stationary
    # lambda_p is the lengthscale of the RBF kernel
    # default 0.6931
    kp_covar_matrix = kp_covariance_function(kp_input, lambda_p = lambda_p)

    ### VIS ###
    # sigma_f is the output variance of the product term
    # Fix cmin and cmax to see sensitivity to output scalar
    fig = make_subplots(rows = 1, cols = 3,
                    subplot_titles = ("Ks", "Kp", "Ks * Kp * sigma_f"))

    fig.add_trace(go.Heatmap(z = ks_covar_matrix.detach().numpy(), 
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                         row = 1, col = 1)

    fig.add_trace(go.Heatmap(z = kp_covar_matrix.detach().numpy(), 
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                         # colorscale = 'haline'),
                         row = 1, col = 2)
    
    fig.add_trace(go.Heatmap(z = ks_covar_matrix.detach().numpy() * kp_covar_matrix.detach().numpy() * sigma_f,
                             # Trade off paramater (ks_covar_matrix.detach().numpy() * (1 - sigma_f)) * (kp_covar_matrix.detach().numpy() * sigma_f)
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                           row = 1, col = 3)

    fig.update_layout(autosize = False, height = 400, width = 900)
    fig.update_yaxes(autorange = "reversed") # matrix style
    fig.update_traces(showscale = False)
    fig.show()

In [ ]:
ks_kp_covar(lambda_s = 0.5, lambda_p = 0.1, sigma_f = 0.8, rowcol_index_tuple = (1, None), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor)
### EXAMPLE ###
# ks_kp_covar(lambda_s = 0.6, lambda_p = 0.08, sigma_f = 0.8, rowcol_index_tuple = (None, 44), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor)
# ks_kp_covar(lambda_s = 0.99, lambda_p = 0.8, sigma_f = 0.5, rowcol_index_tuple = (1, None), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor)
# ks_kp_covar(lambda_s = 0.5, lambda_p = 0.1, sigma_f = 0.8, rowcol_index_tuple = (1, None), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor)

### PARAMETERS ###
#   higher lambda_s: increases receptive spatial region. Cut-off point for zero covariance.
#   lower lambda_p: makes it more sensitive to filter out high pixel correlation values
#   output scaling: Rescaling of output values: 1 will map to [0, 1]

In [ ]:
def covariance_function(kp_ds, lambda_s = 0.3, lambda_p = 0.4, sigma_f = 1.0):

    # Assuming square shape
    dims = kp_ds.shape[1]
    # create input for ks kernel
    # arange function exclude the last one: (1, 46) goes up to (not including) 46
    xs = torch.arange(1, dims + 1).repeat(dims, 1)
    ys = xs.T
    mid_points = torch.cat((ys.unsqueeze(0), xs.unsqueeze(0)), dim = 0)/dims

    ### Ks ### smooth, sparse, local
    # lambda_s is the hp that controls the receptive field
    ks_input = mid_points[:, :, :]
    ks_covar_matrix = ks_covariance_function(tensor = ks_input, lambda_value = lambda_s)


    ### Kp ### coupling(decoupling) of similar(dissimilar) pixel values, non-stationary
    # lambda_p is the lengthscale of the RBF kernel
    # default 0.6931
    kp_input = kp_ds[:, :]
    kp_covar_matrix = kp_covariance_function(kp_input, lambda_p = lambda_p)
    
    # Return components as well
    return (ks_covar_matrix.detach().numpy() * kp_covar_matrix.detach().numpy() * sigma_f), ks_covar_matrix.detach(), kp_covar_matrix.detach()
                        

## Low res

In [ ]:
lr_scene_bed_tensor_norm = minmax_normalise_tensor(lr_scene_bed_tensor)
hr_scene_sur_tensor_norm = minmax_normalise_tensor(hr_scene_sur_tensor)

# Only for visualisation: not used
hr_scene_bed_tensor_norm = minmax_normalise_tensor(hr_scene_bed_tensor)

print(lr_scene_bed_tensor_norm.shape)
print(hr_scene_sur_tensor_norm.shape)
# 9 * 9 * 25
# NH is the number of high-resolution AH areas that compose AL: 25 HR areas compose one LR area (5 x 5)

In [ ]:
fig = px.histogram(lr_scene_bed_tensor_norm.reshape(-1), marginal = "box", nbins = 40)
fig.update_layout(showlegend = False, template = "simple_white", title = "Histogram of lr bed")
fig.show()

In [ ]:
fig = make_subplots(rows = 1, cols = 3,
                    subplot_titles = ("Input: low-res bed", "Input: hih-res surface", "Target: high-res bed"))

fig.add_trace(go.Heatmap(z = lr_scene_bed_tensor.squeeze()[:5, :5], 
                           colorscale = 'haline'),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_scene_bed_tensor.squeeze()[:25, :25], 
                           colorscale = 'haline'),
                           row = 1, col = 3)

fig.add_trace(go.Heatmap(z = hr_scene_sur_tensor.squeeze()[:25, :25], 
                           colorscale = 'gray'),
                           row = 1, col = 2)

fig.update_layout(autosize = False, height = 400, width = 900, title = "Zoom in")
fig.update_layout(plot_bgcolor = 'rgba(0,0,0,0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

In [ ]:
xs = torch.arange(1, 11).repeat(10, 1)
ys = xs.T
# needs to be float
tensor = xs*ys.float()

magnify_2x2blocks = torch.nn.AvgPool2d(kernel_size = (1, 5), stride = (1, 5))   
lr = magnify_2x2blocks(tensor.unsqueeze(0))

### Base covariance

In [ ]:
### Base Covariance ###
# base covariance is based on high-res input only
base_covar, ks, kp = covariance_function(hr_scene_sur_tensor_norm, lambda_s = 0.4, lambda_p = 0.2, sigma_f = 1.0) # torch.Size([2025, 2025])

# Parameter setting
# lambda_s = 0.1 produces stripes (too local, small lengthscale)
# lambda_p = 0.05 produces swirls. Smaller values increase sensitivity (more contrast). High values: Covariance with everything.
# sigma_f = 0.01 keeps it closer to the mean field: Keep at 1.0 to stay in [0, 1] range

In [362]:
fig = make_subplots(rows = 1, cols = 3,
                    subplot_titles = ("ks", "kp", "ks x kp"))

fig.add_trace(go.Heatmap(z = ks[((29 * 45) + 16), :].reshape(45, 45).detach(), 
                           colorscale = 'haline'),
                           row = 1, col = 1)

fig.add_trace(go.Scatter(x = np.array(16.0), y = np.array(29.0), mode = 'markers', 
                         marker = dict(color = "white", size = 16, symbol = 134), showlegend = False),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = kp[((29 * 45) + 16), :].reshape(45, 45).detach(), 
                           colorscale = 'haline'),
                           row = 1, col = 2)

fig.add_trace(go.Scatter(x = np.array(16.0), y = np.array(29.0), mode = 'markers', 
                         marker = dict(color = "white", size = 16, symbol = 134), showlegend = False),
                           row = 1, col = 2)

fig.add_trace(go.Heatmap(z = kp[((29 * 45) + 16)].reshape(45, 45).detach() * ks[((29 * 45) + 16)].reshape(45, 45).detach(), 
                           colorscale = 'haline'),
                           row = 1, col = 3)

fig.add_trace(go.Scatter(x = np.array(16.0), y = np.array(29.0), mode = 'markers', 
                         marker = dict(color = "white", size = 16, symbol = 134), showlegend = False),
                           row = 1, col = 3)

fig.update_layout(autosize = False, height = 400, width = 900)
fig.update_layout(plot_bgcolor = 'rgba(0, 0, 0, 0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
# fig.update_traces(showscale = False)
fig.show()

### lrDict

Create dictionary between lr indices and hr indices.

In [323]:
lrDict = dict()

hr_width = 45
lr_width = 9 
upscaling_factor = 5 # hr_width / lr_width
lr_indices = range(0, 81) # indices of flat lr

lr_row_breaks = np.linspace(0, hr_width**2, num = int(lr_width + 1), dtype = int)

# start with -1 as it will be updated in first it.
lr_row_counter = -1

for i in lr_indices:

    # column counter per row
    j = i%lr_width

    # check for row breaks
    if j == 0:
        lr_row_counter += 1
        row_base = lr_row_breaks[lr_row_counter]

    lr_ranges_list = []
    for w in range(0, upscaling_factor):
        # each list is spanning upscaling_factor rows with upscaling_factor elements each
        lr_ranges_list.extend(range((j * upscaling_factor + (w * hr_width) + row_base), (j * upscaling_factor + (w * hr_width) + 5 + row_base)))

    lrDict[i] = lr_ranges_list

### k_ah_al

Expected output size: torch.Size([2025, 81])  
Pairwise covariance between 2025 (45 x 45) flattened HR pixels and 81 (9 x 9) LR pixels.
N_h = 25 (number of high-resolution AH areas that compose AL, 2025/25 = 81)

Strategies:
- Dictorionary listing all indicies of A_h that make up A_l
- Dual approach with intermediary convolving

In [360]:
# Initialise empty tensor of correct size
k_ah_al = torch.empty(size = (2025, 81))

for i in lr_indices:
    # Fill all rows for each column
    # Average over columns (we are aggregating columns)
    k_ah_al[:, i] = torch.mean(torch.tensor(base_covar[:, lrDict[i]]), dim = 1)

### k_al_al

In [361]:
# Initialise empty tensor of correct size
k_al_al = torch.empty(size = (81, 81))

for i_row in lr_indices:
    for i_col in lr_indices:
        k_al_al[i_row, i_col] = torch.mean(torch.tensor(base_covar[lrDict[i_row], lrDict[i_col]]))

In [363]:
# fix hypers for now
mu = 0.5
noise = 0.05

# Last term: Subtract mean from low-res pixel values. Tensor expressed deviations from mean. Tensor thus ranges from [-0.5, 0.5] and is roughly centred around 0
pl_al_minus_mu = lr_scene_bed_tensor_norm - torch.ones(size = lr_scene_bed_tensor_norm.shape) * mu # torch.Size([1, 9, 9])

# smaller domain: both k_ah_al and k_al_al only occupy 0, 0.1

# Multiply three terms
hr_bed_inf = torch.matmul(k_ah_al,
    torch.matmul(torch.linalg.inv(k_al_al + (torch.eye(n = k_al_al.shape[-1]) * noise)), 
             pl_al_minus_mu.squeeze().reshape(-1))) + mu

# Unadjusted
hr_bed_inf_unadjusted = torch.matmul(k_ah_al,
    torch.matmul(torch.linalg.inv(k_al_al + (torch.eye(n = k_al_al.shape[-1]) * noise)), 
             pl_al_minus_mu.squeeze().reshape(-1))) + mu

# Adjusted
# Reweighting:
W = torch.matmul(k_ah_al, torch.linalg.inv(k_al_al + (torch.eye(n = k_al_al.shape[-1]) * noise))) # torch.Size([2025, 81])
# Neg values

# numerator is torch.Size([2025])
# denominator is torch.Size([2025])
hr_bed_inf_adjusted = torch.div(torch.matmul(W, pl_al_minus_mu.reshape(-1).unsqueeze(1)), torch.matmul(W, torch.ones(size = (81, 1)))) + mu

# Cast into 2D shape
hr_bed_inf_2D = hr_bed_inf_adjusted.reshape(45, -1)

## Normaise/ Reweighting
# Input warping?

In [367]:
fig = make_subplots(rows = 1, cols = 4,
                    subplot_titles = ("low-res bed", "high-res surface", "high-res bed INFERRED", "high-res bed GROUND TRUTH"))

fig.add_trace(go.Heatmap(z = lr_scene_bed_tensor_norm.squeeze(), 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_scene_sur_tensor_norm.squeeze(), 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 2)

# Inference
fig.add_trace(go.Heatmap(z = minmax_normalise_tensor(hr_bed_inf_2D), 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 3)

fig.add_trace(go.Heatmap(z = hr_scene_bed_tensor_norm.squeeze(), 
                           colorscale = 'haline', zmin = 0, zmax = 1),
                           row = 1, col = 4)

# Use difference instead
# fig.add_trace(go.Heatmap(z = hr_scene_bed_tensor_norm.squeeze() - hr_bed_inf_2D, 
#                           colorscale = 'haline', zmin = 0, zmax = 1),
#                           row = 1, col = 4)

fig.update_layout(autosize = False, height = 400, width = 1200)
fig.update_layout(plot_bgcolor = 'rgba(0,0,0,0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

## To Do's

- Other metrics, image fusion

- LML to infer the best parameters
- Create new covariance function
- Clip image values to between 0 and 1.
- Why do we have so many values outside
- look at distributions of variables like base covar

## Done:
- Implement reweighting/pixel correction using low-res data


## Metrics

1. Mean squared error
2. Universal Image Quality index (Wang, Zhou, and Alan C. Bovik. "A universal image quality index." IEEE signal processing letters 9.3 (2002): 81-84.)
   Is this applicable to images which are not rbg?
   - loss of correlation
   - luminance distortion
   - contrast distortion

Look at DeepBedMap paper.

In [368]:
def mse(image1, image2):
    # Take in two tensors of the same size and check t

    # Check
    if (image1.shape != image2.shape):
        print("Input images are not the same size. MSE can't be calculated")

    squared_error_tensor = torch.pow(torch.div(image1.detach(), image2.detach()), exponent = 2)
    mean_squared_error = torch.mean(squared_error_tensor)

    return(mean_squared_error)

In [370]:
mse(hr_scene_bed_tensor_norm, hr_bed_inf_2D)

tensor(1.6072)

In [ ]:
# Domain is off.. check code
print(torch.max(hr_bed_inf_2D))
print(torch.min(hr_bed_inf_2D))